In [ ]:
import polars as pl
from google.colab import files


NULL_VALUES = ["\\N", r"\N", "", " "]


def limpiar_aeropuertos(ruta_entrada: str) -> pl.DataFrame:

    aeropuertos = pl.read_csv(
        ruta_entrada,
        has_header=False,
        new_columns=[
            "id",
            "name",
            "city",
            "country",
            "iata",
            "icao",
            "latitude",
            "longitude",
            "altitude",
            "timezone",
            "dst",
            "tz",
            "type",
            "source"
        ],
        null_values=NULL_VALUES,
        ignore_errors=True
    )

    filas_originales = aeropuertos.height

    aeropuertos_limpios = aeropuertos.select([
        pl.col("id").cast(pl.Int64, strict=False),
        pl.col("name").str.strip_chars(),
        pl.col("city").str.strip_chars(),
        pl.col("country").str.strip_chars(),
        pl.col("iata").str.strip_chars(),
        pl.col("icao").str.strip_chars(),
        pl.col("latitude").cast(pl.Float64, strict=False),
        pl.col("longitude").cast(pl.Float64, strict=False)
    ])

    aeropuertos_sin_nulos = aeropuertos_limpios.drop_nulls([
        "id",
        "name",
        "country",
        "latitude",
        "longitude"
    ])

    filas_sin_nulos = aeropuertos_sin_nulos.height

    aeropuertos_con_coordenadas_validas = aeropuertos_sin_nulos.filter(
        (pl.col("latitude") >= -90) &
        (pl.col("latitude") <= 90) &
        (pl.col("longitude") >= -180) &
        (pl.col("longitude") <= 180)
    )

    filas_coordenadas_validas = aeropuertos_con_coordenadas_validas.height

    aeropuertos_con_codigos_validos = aeropuertos_con_coordenadas_validas.with_columns([
        pl.when(
            pl.col("iata").is_not_null() &
            (pl.col("iata").str.len_chars() == 3)
        )
        .then(pl.col("iata"))
        .otherwise(None)
        .alias("iata"),

        pl.when(
            pl.col("icao").is_not_null() &
            (pl.col("icao").str.len_chars() == 4)
        )
        .then(pl.col("icao"))
        .otherwise(None)
        .alias("icao")
    ])

    aeropuertos_finales = aeropuertos_con_codigos_validos.unique(
        subset=["id"],
        maintain_order=True
    )

    filas_finales = aeropuertos_finales.height

    print("===== REPORTE AEROPUERTOS =====")
    print(f"Filas originales: {filas_originales}")
    print(f"Filas eliminadas por campos críticos nulos: {filas_originales - filas_sin_nulos}")
    print(f"Filas eliminadas por coordenadas inválidas: {filas_sin_nulos - filas_coordenadas_validas}")
    print(f"IDs duplicados eliminados: {filas_coordenadas_validas - filas_finales}")
    print(f"Aeropuertos finales: {filas_finales}")
    print(f"Aeropuertos con IATA: {aeropuertos_finales.filter(pl.col('iata').is_not_null()).height}")
    print(f"Aeropuertos sin IATA: {aeropuertos_finales.filter(pl.col('iata').is_null()).height}")
    print(f"Países únicos: {aeropuertos_finales['country'].n_unique()}")

    return aeropuertos_finales


def limpiar_rutas(ruta_entrada: str) -> pl.DataFrame:
 
    rutas = pl.read_csv(
        ruta_entrada,
        has_header=False,
        new_columns=[
            "airline",
            "airline_id",
            "source_iata",
            "source_id",
            "destination_iata",
            "destination_id",
            "codeshare",
            "stops",
            "equipment"
        ],
        null_values=NULL_VALUES,
        ignore_errors=True
    )

    filas_originales = rutas.height

    rutas_grafo = rutas.select([
        pl.col("source_id").cast(pl.Int64, strict=False),
        pl.col("destination_id").cast(pl.Int64, strict=False)
    ])

    rutas_sin_nulos = rutas_grafo.drop_nulls([
        "source_id",
        "destination_id"
    ])

    filas_sin_nulos = rutas_sin_nulos.height

    rutas_sin_ciclos = rutas_sin_nulos.filter(
        pl.col("source_id") != pl.col("destination_id")
    )

    filas_sin_ciclos = rutas_sin_ciclos.height

    rutas_sin_repetidos = rutas_sin_ciclos.unique(
        subset=["source_id", "destination_id"],
        maintain_order=True
    )

    filas_sin_repetidos = rutas_sin_repetidos.height

    print("\n===== REPORTE RUTAS ANTES DE VALIDAR =====")
    print(f"Filas originales: {filas_originales}")
    print(f"Filas eliminadas por IDs nulos: {filas_originales - filas_sin_nulos}")
    print(f"Self-loops eliminados: {filas_sin_nulos - filas_sin_ciclos}")
    print(f"Rutas repetidas eliminadas: {filas_sin_ciclos - filas_sin_repetidos}")
    print(f"Rutas limpias antes de validar: {filas_sin_repetidos}")
    print(f"Aeropuertos origen únicos: {rutas_sin_repetidos['source_id'].n_unique()}")
    print(f"Aeropuertos destino únicos: {rutas_sin_repetidos['destination_id'].n_unique()}")

    return rutas_sin_repetidos


def validar_rutas_con_aeropuertos(
    aeropuertos: pl.DataFrame,
    rutas: pl.DataFrame
) -> pl.DataFrame:

    rutas_originales = rutas.height

    ids_validos = aeropuertos.select("id")

    rutas_validadas = (
        rutas
        .join(
            ids_validos.rename({"id": "source_id"}),
            on="source_id",
            how="inner"
        )
        .join(
            ids_validos.rename({"id": "destination_id"}),
            on="destination_id",
            how="inner"
        )
    )

    rutas_finales = rutas_validadas.height

    print("\n===== VALIDACIÓN CRUZADA =====")
    print(f"Rutas antes de validar: {rutas_originales}")
    print(f"Rutas eliminadas por aeropuerto inexistente: {rutas_originales - rutas_finales}")
    print(f"Rutas finales válidas: {rutas_finales}")
    print(f"Aeropuertos origen únicos finales: {rutas_validadas['source_id'].n_unique()}")
    print(f"Aeropuertos destino únicos finales: {rutas_validadas['destination_id'].n_unique()}")

    return rutas_validadas


def ejecutar_limpieza_completa() -> None:

    aeropuertos_finales = limpiar_aeropuertos("airports.dat")
    rutas_limpias = limpiar_rutas("routes.dat")

    rutas_validadas = validar_rutas_con_aeropuertos(
        aeropuertos=aeropuertos_finales,
        rutas=rutas_limpias
    )

    aeropuertos_finales.write_csv("airports_clean_final.csv")
    rutas_limpias.write_csv("routes_graph_clean.csv")
    rutas_validadas.write_csv("routes_graph_validated.csv")

    print("\n===== ARCHIVOS GENERADOS =====")
    print("airports_clean_final.csv")
    print("routes_graph_clean.csv")
    print("routes_graph_validated.csv")

    print("\nUsa en C++:")
    print('string airportsFile = "airports_clean_final.csv";')
    print('string routesFile = "routes_graph_validated.csv";')


ejecutar_limpieza_completa()

=== REPORTE DE LIMPIEZA DE RUTAS ===
Filas originales: 67663
Filas eliminadas por IDs faltantes: 423
Self-loops (origen == destino) eliminados: 1
Duplicados eliminados (ID Origen + ID Destino): 29966
Duplicados eliminados (Aerolínea + IDs): 0
------------------------------------
Filas finales en routes_graph_clean.csv: 37273
Filas finales en routes_airline_clean.csv: 67239
Filas finales en routes_full_clean.csv: 67239
------------------------------------
Aerolíneas únicas: 568
Aeropuertos origen únicos: 3315
Aeropuertos destino únicos: 3321


In [ ]:
import polars as pl
from google.colab import files

NULL_VALUES = ["\\N", r"\N", "", " "]


def limpiar_aeropuertos(ruta_entrada: str) -> pl.DataFrame:
    aeropuertos = pl.read_csv(
        ruta_entrada,
        has_header=False,
        new_columns=[
            "id", "name", "city", "country", "iata", "icao",
            "latitude", "longitude", "altitude", "timezone", "dst", "tz", "type", "source"
        ],
        null_values=NULL_VALUES,
        ignore_errors=True
    )

    aeropuertos_limpios = aeropuertos.select([
        pl.col("id").cast(pl.Int64, strict=False),
        pl.col("name").str.strip_chars(),
        pl.col("city").str.strip_chars(),
        pl.col("country").str.strip_chars(),
        pl.col("iata").str.strip_chars(),
        pl.col("icao").str.strip_chars(),
        pl.col("latitude").cast(pl.Float64, strict=False),
        pl.col("longitude").cast(pl.Float64, strict=False)
    ]).drop_nulls(["id", "name", "country", "latitude", "longitude"])

    aeropuertos_finales = aeropuertos_limpios.filter(
        (pl.col("latitude") >= -90) & (pl.col("latitude") <= 90) &
        (pl.col("longitude") >= -180) & (pl.col("longitude") <= 180)
    ).unique(subset=["id"], maintain_order=True)

    print("===== REPORTE AEROPUERTOS =====")
    print(f"Aeropuertos limpios listos: {aeropuertos_finales.height}\n")
    return aeropuertos_finales


def fabricar_catalogo_aerolineas(ruta_routes: str) -> pl.DataFrame:
   
    rutas_raw = pl.read_csv(
        ruta_routes,
        has_header=False,
        new_columns=[
            "airline_code", "airline_id", "source_iata", "source_id",
            "destination_iata", "destination_id", "codeshare", "stops", "equipment"
        ],
        null_values=NULL_VALUES,
        ignore_errors=True
    )

    filas_originales = rutas_raw.height

    aerolineas_extraidas = rutas_raw.select([
        pl.col("airline_id").cast(pl.Int64, strict=False),
        pl.col("airline_code").str.strip_chars()
    ]).drop_nulls(["airline_id", "airline_code"])

    aerolineas_finales = aerolineas_extraidas.unique(
        subset=["airline_id"],
        maintain_order=True
    )

    print("===== REPORTE AEROLÍNEAS (GENERACIÓN AUTOMÁTICA) =====")
    print(f"Rutas analizadas para extracción: {filas_originales}")
    print(f"Catálogo maestro de aerolíneas generado con éxito: {aerolineas_finales.height} operadoras únicas.\n")

    return aerolineas_finales


def limpiar_rutas_v5(ruta_entrada: str) -> pl.DataFrame:
    rutas = pl.read_csv(
        ruta_entrada,
        has_header=False,
        new_columns=[
            "airline_code", "airline_id", "source_iata", "source_id",
            "destination_iata", "destination_id", "codeshare", "stops", "equipment"
        ],
        null_values=NULL_VALUES,
        ignore_errors=True
    )

    rutas_grafo = rutas.select([
        pl.col("source_id").cast(pl.Int64, strict=False),
        pl.col("destination_id").cast(pl.Int64, strict=False),
        pl.col("airline_id").cast(pl.Int64, strict=False)
    ]).drop_nulls(["source_id", "destination_id", "airline_id"])

    rutas_sin_ciclos = rutas_grafo.filter(pl.col("source_id") != pl.col("destination_id"))

    rutas_finales = rutas_sin_ciclos.unique(
        subset=["source_id", "destination_id", "airline_id"],
        maintain_order=True
    )

    print("===== REPORTE RUTAS PRE-VALIDACIÓN =====")
    print(f"Rutas base del multígrafo listas: {rutas_finales.height}\n")
    return rutas_finales


def validar_integridad_referencial_v5(
    aeropuertos: pl.DataFrame,
    aerolineas: pl.DataFrame,
    rutas: pl.DataFrame
) -> pl.DataFrame:
    """Valida de forma cruzada que existan origen, destino y aerolínea."""
    ids_aeropuertos_validos = aeropuertos.select("id")
    ids_aerolineas_validas = aerolineas.select("airline_id")

    rutas_validadas = (
        rutas
        .join(ids_aeropuertos_validos.rename({"id": "source_id"}), on="source_id", how="inner")
        .join(ids_aeropuertos_validos.rename({"id": "destination_id"}), on="destination_id", how="inner")
        .join(ids_aerolineas_validas, on="airline_id", how="inner")
    )

    print("===== VALIDACIÓN DE INTEGRIDAD REFERENCIAL (CRUZADA) =====")
    print(f"Aristas finales del Multígrafo: {rutas_validadas.height}")
    print(f"Aerolíneas validadas volando activamente: {rutas_validadas['airline_id'].n_unique()}\n")

    return rutas_validadas


def ejecutar_pipeline_reto5() -> None:
    print("Iniciando Pipeline de Ingeniería de Datos para Reto 5 (Modo Inverso)...\n")

    aeropuertos_df = limpiar_aeropuertos("airports.dat")

    aerolineas_df = fabricar_catalogo_aerolineas("routes.dat")

    rutas_df = limpiar_rutas_v5("routes.dat")

    rutas_validadas_df = validar_integridad_referencial_v5(
        aeropuertos=aeropuertos_df,
        aerolineas=aerolineas_df,
        rutas=rutas_df
    )

    aeropuertos_df.write_csv("airports_clean_final.csv")
    aerolineas_df.write_csv("airlines_clean_final.csv")  # <-- Tu nuevo catálogo fabricado
    rutas_validadas_df.write_csv("routes_multigraph_v5.csv")

    print("===== PROCESO COMPLETADO =====")
    print("Archivos listos para C++:")
    print("1. airports_clean_final.csv")
    print("2. airlines_clean_final.csv  (Contiene: airline_id, airline_code)")
    print("3. routes_multigraph_v5.csv   (Contiene: source_id, destination_id, airline_id)")


ejecutar_pipeline_reto5()